# Parkinson's Disease Detection from Spiral & Wave Drawings (Google Colab)

**Dataset:** Kaggle `kmader/parkinsons-drawings` (downloaded automatically via `kagglehub` — no manual upload or API key required).  
**How to run:** Enable GPU (**Runtime -> Change runtime type -> T4 GPU**) and click **Runtime -> Run all**.

### Unique & Efficient Features
1. **Automatic Mirrored-Folder Deduplication & Stratified Splitting:** Cleans mirrored folders in the Kaggle archive before splitting `train` / `valid` / `test`.
2. **Leakage & Shortcut Auditing:** Uses **MD5** + **DCT Perceptual Hashing (`pHash`)** to eliminate cross-split near-duplicates and runs a **Welch's *t*-test / Cohen's *d*** audit for brightness shortcuts.
3. **Zero-Double-Scaling `tf.data` Pipeline:** Caches `[0, 255]` tensors *before* GPU-vectorized, drawing-safe augmentation (no vertical flips), while each model applies its own native scaling.
4. **3-Model Benchmark + Clinical Calibration:** Compares **Scratch CNN**, **VGG16**, and **ResNet50** (2-phase transfer learning with frozen `BatchNormalization`), plus **Youden's *J* optimal recall threshold**.
5. **Keras 3-Safe Grad-CAM + Live Gradio App:** Visualizes what stroke regions drive each prediction and launches an interactive screening demo.

In [ ]:
# @title 0. Environment Setup, Reproducibility & GPU Check
!pip -q install kagglehub gradio

import os
import time
import json
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from scipy import stats

import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10
})
PALETTE = {"healthy": "#2A9D8F", "parkinson": "#E76F51"}

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow: {tf.__version__} | GPUs detected: {len(gpus)}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# @title 1. Download Kaggle Dataset & Build Clean Metadata DataFrame
path = kagglehub.dataset_download("kmader/parkinsons-drawings")
ROOT = Path(path)
print("Dataset path:", ROOT)

SPLITS = ["train", "valid", "test"]
CLASSES = ["healthy", "parkinson"]
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CLASS_ALIASES = {
    "healthy": "healthy", "control": "healthy", "normal": "healthy",
    "parkinson": "parkinson", "parkinsons": "parkinson", "pd": "parkinson", "patient": "parkinson"
}
SPLIT_ALIASES = {
    "train": "train", "training": "train",
    "valid": "valid", "val": "valid", "validation": "valid",
    "test": "test", "testing": "test"
}

records = []
for file in sorted(ROOT.rglob("*")):
    if not (file.is_file() and file.suffix.lower() in VALID_EXTS):
        continue
    if any(part.startswith(".") or part == "__MACOSX" for part in file.parts):
        continue
    parts_lower = [p.lower() for p in file.relative_to(ROOT).parts[:-1]]
    detected_cls = next((CLASS_ALIASES[p] for p in reversed(parts_lower) if p in CLASS_ALIASES), None)
    detected_split = next((SPLIT_ALIASES[p] for p in reversed(parts_lower) if p in SPLIT_ALIASES), None)
    drawing_type = "spiral" if "spiral" in parts_lower else ("wave" if "wave" in parts_lower else "unknown")
    if detected_cls is not None:
        records.append({
            "path": str(file),
            "split": detected_split,
            "label": detected_cls,
            "label_idx": CLASSES.index(detected_cls),
            "drawing_type": drawing_type
        })

df = pd.DataFrame(records)
assert len(df) > 0, f"No labeled images found under {ROOT}."

# Deduplicate mirrored folders (e.g. drawings/spiral vs spiral) before creating validation split
df["_md5"] = [hashlib.md5(Path(p).read_bytes()).hexdigest() for p in df["path"]]
before_dedup = len(df)
df = df.drop_duplicates(subset=["_md5"], keep="first").drop(columns=["_md5"]).reset_index(drop=True)
if len(df) < before_dedup:
    print(f"Removed {before_dedup - len(df)} mirrored duplicate file paths.")

# Carve stratified validation split (18% of training set) if not provided
if "valid" not in df["split"].values:
    tr_sub = df[df["split"] == "train"]
    strat_key = tr_sub["label"] + "_" + tr_sub["drawing_type"]
    _, va_idx = train_test_split(tr_sub.index, test_size=0.18, stratify=strat_key, random_state=SEED)
    df.loc[va_idx, "split"] = "valid"
    print("Created stratified validation split (18% of train, balanced across spiral & wave).")

print(f"Loaded {len(df)} unique images.")
df.head()

In [ ]:
# @title 2 & 3. Class Distribution, Imbalance Check, Image Sizes & Channels
print("Total images:", len(df))
split_counts = df.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(SPLITS)
print("\nCounts per split per class:\n", split_counts)
print("\nCounts per drawing type:\n", df.groupby(["drawing_type", "label"]).size().unstack(fill_value=0))

class_counts = df["label"].value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nOverall class counts:\n{class_counts.to_string()}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
USE_CLASS_WEIGHTS = imbalance_ratio > 1.5

widths, heights, channels_list, formats_list = [], [], [], []
mode_to_ch = {"L": 1, "P": 1, "RGB": 3, "RGBA": 4, "CMYK": 4}

for p in df["path"]:
    try:
        with Image.open(p) as img:
            w, h = img.size
            ch = mode_to_ch.get(img.mode, len(img.getbands()))
            fmt = img.format or Path(p).suffix.upper().lstrip(".")
    except Exception:
        w, h, ch, fmt = np.nan, np.nan, -1, "CORRUPT"
    widths.append(w)
    heights.append(h)
    channels_list.append(ch)
    formats_list.append(fmt)

df["width"] = widths
df["height"] = heights
df["channels"] = channels_list
df["format"] = formats_list

print("\nImage Dimension Summary:\n", df[["width", "height"]].describe().round(1))
print("Unique channels:", df["channels"].unique(), "| Formats:", df["format"].value_counts().to_dict())

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
class_counts.reindex(CLASSES).plot(kind="bar", color=[PALETTE[c] for c in CLASSES], ax=axes[0, 0])
axes[0, 0].set_title(f"Overall Class Balance (Ratio = {imbalance_ratio:.2f})")
axes[0, 0].set_ylabel("Image Count")
axes[0, 0].tick_params(axis="x", rotation=0)

split_counts[CLASSES].plot(kind="bar", color=[PALETTE[c] for c in CLASSES], ax=axes[0, 1])
axes[0, 1].set_title("Split-by-Class Distribution")
axes[0, 1].set_ylabel("Image Count")
axes[0, 1].tick_params(axis="x", rotation=0)

axes[1, 0].hist(df["width"].dropna(), bins=20, alpha=0.65, label="Width", color="#264653")
axes[1, 0].hist(df["height"].dropna(), bins=20, alpha=0.65, label="Height", color="#E9C46A")
axes[1, 0].set_title("Width & Height Distribution")
axes[1, 0].set_xlabel("Pixels")
axes[1, 0].legend()

for cls in CLASSES:
    sub = df[df["label"] == cls]
    axes[1, 1].scatter(sub["width"], sub["height"], alpha=0.65, label=cls, color=PALETTE[cls], edgecolors="white", s=45)
axes[1, 1].set_title("Image Resolution Scatter by Class")
axes[1, 1].set_xlabel("Width (px)")
axes[1, 1].set_ylabel("Height (px)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("eda_overview.png", bbox_inches="tight")
plt.show()

TARGET_SIZE = (224, 224)
print(f"Standardizing all images to {TARGET_SIZE} x 3 RGB channels.")

In [ ]:
# @title 4 & 5. Sample Grid + Pixel Intensity & Shortcut Learning Audit
N_COLS = 8
fig, axes = plt.subplots(len(CLASSES), N_COLS, figsize=(2.2 * N_COLS, 2.5 * len(CLASSES)))

for r, cls in enumerate(CLASSES):
    pool = df[(df["label"] == cls) & (df["channels"] > 0)]
    samples = pool.sample(n=min(N_COLS, len(pool)), random_state=SEED)
    for c, (_, row) in enumerate(samples.iterrows()):
        img = ImageOps.exif_transpose(Image.open(row["path"])).convert("RGB")
        axes[r, c].imshow(img)
        axes[r, c].set_title(f"{row['label']} ({row['drawing_type']})\n{int(row['width'])}x{int(row['height'])}", fontsize=8)
        axes[r, c].axis("off")

plt.suptitle("Random Sample Grid Across Classes", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_grid.png", bbox_inches="tight")
plt.show()

# Pixel intensity distribution & automated shortcut-learning audit
plt.figure(figsize=(10, 4))
class_means = {}
for cls in CLASSES:
    pool = df[(df["label"] == cls) & (df["channels"] > 0)]
    sample_rows = pool.sample(n=min(50, len(pool)), random_state=SEED)
    pixels, img_means = [], []
    for p in sample_rows["path"]:
        arr = np.asarray(Image.open(p).convert("RGB").resize(TARGET_SIZE), dtype=np.float32)
        pixels.append(arr.ravel())
        img_means.append(arr.mean())
    class_means[cls] = np.array(img_means)
    plt.hist(np.concatenate(pixels), bins=256, range=(0, 255), density=True, alpha=0.45, label=cls, color=PALETTE[cls])

plt.title("Pixel Intensity Distribution by Class (Shortcut Audit)")
plt.xlabel("Pixel Intensity (0-255)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig("pixel_intensity_audit.png", bbox_inches="tight")
plt.show()

train_subset = df[(df["split"] == "train") & (df["channels"] > 0)]
channel_sum = np.zeros(3, dtype=np.float64)
channel_sq_sum = np.zeros(3, dtype=np.float64)
n_pixels = 0

for p in train_subset["path"]:
    arr = np.asarray(Image.open(p).convert("RGB").resize(TARGET_SIZE), dtype=np.float64) / 255.0
    channel_sum += arr.sum(axis=(0, 1))
    channel_sq_sum += (arr ** 2).sum(axis=(0, 1))
    n_pixels += arr.shape[0] * arr.shape[1]

train_mean = channel_sum / n_pixels
train_std = np.sqrt(channel_sq_sum / n_pixels - train_mean ** 2)
print(f"Training Set RGB Mean (0-1): {np.round(train_mean, 4)}")
print(f"Training Set RGB Std  (0-1): {np.round(train_std, 4)}")

t_stat, p_val = stats.ttest_ind(class_means["healthy"], class_means["parkinson"], equal_var=False)
pooled_std = np.sqrt((class_means["healthy"].var(ddof=1) + class_means["parkinson"].var(ddof=1)) / 2)
cohens_d = abs(class_means["healthy"].mean() - class_means["parkinson"].mean()) / (pooled_std + 1e-8)
print(f"Brightness Shortcut Check -> Cohen's d = {cohens_d:.3f}, p = {p_val:.4f}")
if cohens_d > 0.5 and p_val < 0.05:
    print("WARNING: Significant brightness difference between classes! Brightness/contrast augmentation is essential.")

In [ ]:
# @title 6. Corrupt Image Check + Exact (MD5) & Near-Duplicate (pHash) Leakage Removal
corrupt_paths = []
for p in df["path"]:
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_paths.append(p)

print(f"Corrupt images found: {len(corrupt_paths)}")
if corrupt_paths:
    df = df[~df["path"].isin(corrupt_paths)].reset_index(drop=True)

def compute_md5(path: str) -> str:
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

def compute_phash(path: str, hash_size: int = 8) -> str:
    from scipy.fftpack import dct
    img = Image.open(path).convert("L").resize((hash_size * 4, hash_size * 4), Image.Resampling.LANCZOS)
    pixels = np.asarray(img, dtype=np.float32)
    dct_2d = dct(dct(pixels.T, norm="ortho").T, norm="ortho")
    low_freq = dct_2d[:hash_size, :hash_size]
    median = np.median(low_freq[1:, 1:])
    bits = (low_freq > median).flatten()
    return "".join("1" if b else "0" for b in bits)

def hamming_dist(h1: str, h2: str) -> int:
    return sum(c1 != c2 for c1, c2 in zip(h1, h2))

df["md5"] = [compute_md5(p) for p in df["path"]]
df["phash"] = [compute_phash(p) for p in df["path"]]

split_priority = {"train": 0, "valid": 1, "test": 2}
df["priority"] = df["split"].map(split_priority)
df = df.sort_values("priority").reset_index(drop=True)

exact_dupes = df[df.duplicated(subset=["md5"], keep="first")]
cross_split_exact = exact_dupes[exact_dupes["split"].isin(["valid", "test"])]

PHASH_THRESHOLD = 4
train_rows = df[df["split"] == "train"]
near_leak_indices = []
for idx, row in df[df["split"].isin(["valid", "test"])].iterrows():
    if idx in cross_split_exact.index:
        continue
    for _, tr in train_rows.iterrows():
        if hamming_dist(row["phash"], tr["phash"]) <= PHASH_THRESHOLD:
            near_leak_indices.append(idx)
            break

print(f"Exact duplicates removed from valid/test: {len(cross_split_exact)}")
print(f"Near-duplicates (pHash <= {PHASH_THRESHOLD}) removed from valid/test: {len(near_leak_indices)}")

drop_indices = set(cross_split_exact.index).union(near_leak_indices)
if drop_indices:
    df = df.drop(index=list(drop_indices)).reset_index(drop=True)

train_df = df[df["split"] == "train"].reset_index(drop=True)
valid_df = df[df["split"] == "valid"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)
print(f"Clean Split Sizes -> Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

In [ ]:
# @title 7 & 8. Unified tf.data Pipeline + Drawing-Safe Augmentation
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Keep images in [0, 255] float32 so each model applies its own native scaling inside its graph.
def preprocess_image(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, TARGET_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)

# Label-preserving transforms for drawings (no vertical flip or large rotation)
augment_layer = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(10.0 / 360.0, fill_mode="nearest", seed=SEED),
    layers.RandomZoom(height_factor=(-0.1, 0.1), width_factor=(-0.1, 0.1), fill_mode="nearest", seed=SEED),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1, fill_mode="nearest", seed=SEED),
    layers.RandomBrightness(factor=0.1, value_range=(0.0, 255.0), seed=SEED),
    layers.RandomContrast(factor=0.1, seed=SEED),
], name="train_augmentation")

def build_pipeline(df_split: pd.DataFrame, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((df_split["path"].values, df_split["label_idx"].values))
    ds = ds.map(preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()  # cache decoded [0,255] images before shuffle & augment
    if training:
        ds = ds.shuffle(buffer_size=max(len(df_split), 1), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(
            lambda x, y: (tf.clip_by_value(augment_layer(x, training=True), 0.0, 255.0), y),
            num_parallel_calls=AUTOTUNE
        )
    return ds.prefetch(AUTOTUNE)

train_ds = build_pipeline(train_df, training=True)
val_ds   = build_pipeline(valid_df, training=False)
test_ds  = build_pipeline(test_df, training=False)

# Sanity-check: 9 augmented versions of 1 training drawing
sample_img, sample_lbl = next(iter(build_pipeline(train_df.iloc[:1], training=False)))
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax in axes.ravel():
    aug_img = tf.clip_by_value(augment_layer(sample_img, training=True), 0.0, 255.0)[0].numpy().astype("uint8")
    ax.imshow(aug_img)
    ax.axis("off")
plt.suptitle(f"Augmentation Sanity Check (Class: {CLASSES[int(sample_lbl[0])]})", fontweight="bold")
plt.tight_layout()
plt.savefig("augmentation_sanity_check.png", bbox_inches="tight")
plt.show()

In [ ]:
# @title 9. Modeling Setup, Class Weights & Callbacks
MODELS = {}
BASE_MODELS = {}
HISTORY = {}
RESULTS = {}
TRAIN_TIMES = {}

EPOCHS = 30
os.makedirs("checkpoints", exist_ok=True)
CKPT_EXT = ".weights.h5" if int(tf.__version__.split(".")[1]) >= 16 else ".h5"

y_train_vals = train_df["label_idx"].values
weights_arr = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_train_vals)
CLASS_WEIGHTS = {0: float(weights_arr[0]), 1: float(weights_arr[1])} if USE_CLASS_WEIGHTS else None
print("Class weights used:", CLASS_WEIGHTS)

def get_callbacks(model_name: str, patience_es: int = 5):
    return [
        callbacks.EarlyStopping(monitor="val_loss", patience=patience_es, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        callbacks.ModelCheckpoint(f"checkpoints/{model_name}{CKPT_EXT}", monitor="val_loss", save_best_only=True, save_weights_only=True)
    ]

def merge_histories(h1, h2):
    merged = {k: list(h1.history[k]) + list(h2.history.get(k, [])) for k in h1.history}
    merged["finetune_start_epoch"] = len(h1.history["loss"])
    return merged

In [ ]:
# @title 10. Model 1 — Custom CNN From Scratch
def build_cnn():
    return models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Rescaling(1.0 / 255.0),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ], name="Scratch_CNN")

cnn = build_cnn()
cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

t0 = time.perf_counter()
h_cnn = cnn.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks("Scratch_CNN"), class_weight=CLASS_WEIGHTS, verbose=1
)
TRAIN_TIMES["Scratch_CNN"] = time.perf_counter() - t0
MODELS["Scratch_CNN"] = cnn
HISTORY["Scratch_CNN"] = dict(h_cnn.history)
print(f"Scratch_CNN trained in {TRAIN_TIMES['Scratch_CNN']:.1f}s")

In [ ]:
# @title 11. Models 2 & 3 — Two-Phase Transfer Learning (VGG16 & ResNet50)
class ImageNetCaffePreprocess(layers.Layer):
    def call(self, inputs):
        x = tf.cast(inputs, tf.float32)[..., ::-1]  # RGB -> BGR
        return x - tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)

BACKBONE_FACTORY = {
    "VGG16": tf.keras.applications.VGG16,
    "ResNet50": tf.keras.applications.ResNet50,
}

def build_transfer(base_name: str):
    base = BACKBONE_FACTORY[base_name](weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    inp = layers.Input(shape=(224, 224, 3))
    x = ImageNetCaffePreprocess(name="caffe_preprocess")(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.5, name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return models.Model(inp, out, name=base_name), base

for name in ["VGG16", "ResNet50"]:
    print(f"\n{'='*55}\nTraining {name} (Phase 1: Head -> Phase 2: Fine-Tune)\n{'='*55}")
    model, base = build_transfer(name)
    t0 = time.perf_counter()

    model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=get_callbacks(f"{name}_p1"), class_weight=CLASS_WEIGHTS, verbose=1)

    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=get_callbacks(f"{name}_p2"), class_weight=CLASS_WEIGHTS, verbose=1)

    TRAIN_TIMES[name] = time.perf_counter() - t0
    MODELS[name] = model
    BASE_MODELS[name] = base
    HISTORY[name] = merge_histories(h1, h2)
    print(f"{name} completed in {TRAIN_TIMES[name]:.1f}s")

In [ ]:
# @title 12. Evaluate All Models on Test Set + Clinical Threshold Calibration
y_true = np.concatenate([y.numpy() for _, y in test_ds]).astype(int)
val_true = np.concatenate([y.numpy() for _, y in val_ds]).astype(int)

def find_optimal_threshold(y_val_true, y_val_prob):
    if len(np.unique(y_val_true)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_val_true, y_val_prob)
    best_idx = int(np.argmax(tpr - fpr))
    return float(np.clip(thresholds[best_idx], 0.1, 0.9))

def evaluate_model(name, model, test_ds, val_ds):
    y_prob = model.predict(test_ds, verbose=0).ravel()
    val_prob = model.predict(val_ds, verbose=0).ravel()
    opt_thresh = find_optimal_threshold(val_true, val_prob)

    y_pred = (y_prob >= 0.5).astype(int)
    y_pred_opt = (y_prob >= opt_thresh).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "auc": roc_auc_score(y_true, y_prob),
        "opt_threshold": opt_thresh,
        "opt_recall": recall_score(y_true, y_pred_opt, zero_division=0),
        "conf_matrix": cm,
        "fpr_tpr": (fpr, tpr),
        "y_prob": y_prob,
        "y_pred": y_pred,
        "params": model.count_params(),
        "train_time": TRAIN_TIMES[name],
    }

for name, model in MODELS.items():
    RESULTS[name] = evaluate_model(name, model, test_ds, val_ds)

metric_cols = ["accuracy", "precision", "recall", "f1", "specificity", "auc", "opt_threshold", "opt_recall", "params", "train_time"]
table = pd.DataFrame({name: {k: RESULTS[name][k] for k in metric_cols} for name in MODELS}).T
display(table.round(3))

best_model_name = table["f1"].idxmax()
highest_recall_name = table["recall"].idxmax()
print(f"Best overall (F1): {best_model_name} | Highest Parkinson's Recall: {highest_recall_name}")

In [ ]:
# @title 13. Visual Comparison Dashboard (Metrics, Confusion Matrices, ROC, Curves)
model_names = list(MODELS.keys())

ax = table[["accuracy", "precision", "recall", "f1"]].plot(kind="bar", figsize=(9, 4.5), colormap="viridis")
ax.set_title("Model Comparison on Held-Out Test Set", fontweight="bold")
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig("metrics_comparison.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(model_names), figsize=(4.5 * len(model_names), 4))
for ax, name in zip(axes, model_names):
    sns.heatmap(RESULTS[name]["conf_matrix"], annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, cbar=False, ax=ax)
    ax.set_title(f"{name} (F1={RESULTS[name]['f1']:.2f})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(6.5, 5.5))
for name in model_names:
    fpr, tpr = RESULTS[name]["fpr_tpr"]
    plt.plot(fpr, tpr, lw=2.2, label=f"{name} (AUC = {RESULTS[name]['auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Baseline")
plt.title("ROC Curves — Parkinson's Screening", fontweight="bold")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Recall / Sensitivity)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, len(model_names), figsize=(5 * len(model_names), 7))
for col, name in enumerate(model_names):
    h = HISTORY[name]
    epochs_x = range(1, len(h["loss"]) + 1)
    axes[0, col].plot(epochs_x, h["loss"], label="Train Loss")
    axes[0, col].plot(epochs_x, h["val_loss"], label="Val Loss")
    axes[1, col].plot(epochs_x, h["accuracy"], label="Train Acc")
    axes[1, col].plot(epochs_x, h["val_accuracy"], label="Val Acc")
    if "finetune_start_epoch" in h:
        ft_ep = h["finetune_start_epoch"] + 0.5
        axes[0, col].axvline(ft_ep, color="gray", ls="--", alpha=0.7, label="Fine-tune")
        axes[1, col].axvline(ft_ep, color="gray", ls="--", alpha=0.7, label="Fine-tune")
    axes[0, col].set_title(f"{name} — Loss")
    axes[1, col].set_title(f"{name} — Accuracy")
    axes[1, col].set_xlabel("Epoch")
    axes[0, col].legend(fontsize=8)
    axes[1, col].legend(fontsize=8)

plt.tight_layout()
plt.savefig("training_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# @title 14. Error Analysis & Keras 3-Safe Grad-CAM Explainability
test_images = np.concatenate([x.numpy() for x, _ in test_ds], axis=0)

for name in model_names:
    y_pred = RESULTS[name]["y_pred"]
    y_prob = RESULTS[name]["y_prob"]
    wrong_idx = np.where(y_pred != y_true)[0]
    print(f"{name}: {len(wrong_idx)} / {len(y_true)} test errors")
    if len(wrong_idx) == 0:
        continue
    show_idx = wrong_idx[:8]
    fig, axes = plt.subplots(1, len(show_idx), figsize=(2.4 * len(show_idx), 2.8))
    if len(show_idx) == 1:
        axes = [axes]
    for ax, idx in zip(axes, show_idx):
        ax.imshow(test_images[idx].astype("uint8"))
        ax.set_title(f"Pred: {CLASSES[y_pred[idx]]} ({y_prob[idx]:.2f})\nTrue: {CLASSES[y_true[idx]]}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Misclassified Test Samples — {name}", fontweight="bold")
    plt.tight_layout()
    plt.show()

def compute_gradcam(img_array_255: np.ndarray, model_name: str) -> np.ndarray:
    img_tensor = tf.cast(tf.expand_dims(img_array_255, axis=0), tf.float32)
    model = MODELS[model_name]

    if model_name == "Scratch_CNN":
        last_conv_idx = max(i for i, l in enumerate(model.layers) if isinstance(l, layers.Conv2D))
        with tf.GradientTape() as tape:
            x = img_tensor
            for l in model.layers[:last_conv_idx + 1]:
                x = l(x, training=False)
            conv_out = x
            tape.watch(conv_out)
            for l in model.layers[last_conv_idx + 1:]:
                x = l(x, training=False)
            pred = x[:, 0]
    else:
        base = BASE_MODELS[model_name]
        preferred = {"VGG16": "block5_conv3", "ResNet50": "conv5_block3_out"}.get(model_name)
        last_conv_name = (
            preferred if preferred and preferred in [l.name for l in base.layers]
            else next(l.name for l in reversed(base.layers) if isinstance(l, layers.Conv2D))
        )
        conv_extractor = tf.keras.Model(base.input, [base.get_layer(last_conv_name).output, base.output])
        preprocess_layer = model.get_layer("caffe_preprocess")
        gap_layer = model.get_layer("gap")
        head_dense = model.get_layer("classifier")

        with tf.GradientTape() as tape:
            pre = preprocess_layer(img_tensor)
            conv_out, base_out = conv_extractor(pre, training=False)
            tape.watch(conv_out)
            pred = head_dense(gap_layer(base_out))[:, 0]

    grads = tape.gradient(pred, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_out[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return tf.image.resize(heatmap[..., tf.newaxis], TARGET_SIZE).numpy().squeeze()

def overlay_gradcam(img_255: np.ndarray, heatmap: np.ndarray, alpha: float = 0.42) -> np.ndarray:
    cmap = plt.get_cmap("jet")(heatmap)[..., :3] * 255.0
    return np.clip((1 - alpha) * img_255 + alpha * cmap, 0, 255).astype("uint8")

best_res = RESULTS[best_model_name]
wrong_best = np.where(best_res["y_pred"] != y_true)[0].tolist()
right_best = np.where(best_res["y_pred"] == y_true)[0].tolist()
vis_indices = (wrong_best[:3] + right_best[:6])[:6]

fig, axes = plt.subplots(2, len(vis_indices), figsize=(3 * len(vis_indices), 6))
if len(vis_indices) == 1:
    axes = axes[:, np.newaxis]
for c, idx in enumerate(vis_indices):
    img = test_images[idx]
    hm = compute_gradcam(img, best_model_name)
    axes[0, c].imshow(img.astype("uint8"))
    axes[0, c].set_title(f"True: {CLASSES[y_true[idx]]}", fontsize=9)
    axes[0, c].axis("off")
    axes[1, c].imshow(overlay_gradcam(img, hm))
    axes[1, c].set_title(f"Pred: {CLASSES[best_res['y_pred'][idx]]} ({best_res['y_prob'][idx]:.2f})", fontsize=9)
    axes[1, c].axis("off")

plt.suptitle(f"Grad-CAM Attention Maps — {best_model_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# @title 15. Final Summary Report + Export JSON
print("=" * 65)
print("PARKINSON'S DISEASE DETECTION — FINAL EVALUATION REPORT")
print("=" * 65)
print(f"Best model (by F1):           {best_model_name}")
print(f"Recall for parkinson (0.5):   {RESULTS[best_model_name]['recall']:.3f}")
print(f"Recall at optimal threshold:  {RESULTS[best_model_name]['opt_recall']:.3f} (thresh = {RESULTS[best_model_name]['opt_threshold']:.3f})")
print(f"ROC-AUC:                      {RESULTS[best_model_name]['auc']:.3f}")
print(f"Specificity:                  {RESULTS[best_model_name]['specificity']:.3f}")
print("-" * 65)
print(f"Note: Small dataset ({len(df)} total images), so report test metrics with caution.")
print("Medical Disclaimer: Screening aid only — not a substitute for clinical diagnosis.")

clean_counts = {f"{sp}/{lb}": int(cnt) for (sp, lb), cnt in df.groupby(["split", "label"]).size().items()}
summary_export = {
    "best_model": best_model_name,
    "dataset_clean_counts": clean_counts,
    "brightness_cohens_d": float(cohens_d),
    "metrics": table.round(4).to_dict(orient="index")
}
with open("parkinson_report.json", "w") as f:
    json.dump(summary_export, f, indent=2, default=str)
print("Saved parkinson_report.json")

In [ ]:
# @title 16. (Optional) 5-Fold Stratified Cross-Validation
RUN_CV = False  # Set True to run 5-fold CV

if RUN_CV:
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []
    cv_base = best_model_name if best_model_name in BACKBONE_FACTORY else "ResNet50"

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["label_idx"]), 1):
        tr_ds = build_pipeline(df.iloc[tr_idx].reset_index(drop=True), training=True)
        va_ds = build_pipeline(df.iloc[va_idx].reset_index(drop=True), training=False)
        m, _ = build_transfer(cv_base)
        m.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
        m.fit(tr_ds, validation_data=va_ds, epochs=12, callbacks=[callbacks.EarlyStopping(patience=4, restore_best_weights=True)], verbose=0)
        y_va = df.iloc[va_idx]["label_idx"].values
        p_va = (m.predict(va_ds, verbose=0).ravel() >= 0.5).astype(int)
        f1_f = f1_score(y_va, p_va, zero_division=0)
        rec_f = recall_score(y_va, p_va, zero_division=0)
        cv_scores.append((f1_f, rec_f))
        print(f"Fold {fold}: F1 = {f1_f:.3f} | Recall = {rec_f:.3f}")

    arr = np.array(cv_scores)
    print(f"5-Fold {cv_base} -> F1: {arr[:,0].mean():.3f} +/- {arr[:,0].std():.3f} | Recall: {arr[:,1].mean():.3f} +/- {arr[:,1].std():.3f}")

In [ ]:
# @title 17. Full-Screen 3D DNA/Neuron Background + Exact Accuracy, Precision, Recall & F1 UI
import gradio as gr

# Default to best_model_name @ 0.50 so UI metrics match Cell 12's test table 100% identically
DEFAULT_THRESH = 0.50
OPT_THRESH = round(float(RESULTS[best_model_name]["opt_threshold"]), 2)
MODEL_CHOICES = [best_model_name] + [m for m in MODELS.keys() if m != best_model_name] + ["Ensemble (All 3 Models)"]

example_paths = []
for cls in CLASSES:
    sub = test_df[test_df["label"] == cls]
    if len(sub) > 0:
        example_paths.extend([[p] for p in sub["path"].head(2).tolist()])

def make_bg_helix(side_class: str, n_rungs: int = 24) -> str:
    rungs = ""
    for i in range(n_rungs):
        delay = round(-i * 0.26, 2)
        rungs += f'<div class="bg-dna-rung" style="animation-delay:{delay}s;"><span class="bg-nuc-l"></span><span class="bg-bond"></span><span class="bg-nuc-r"></span></div>'
    return f'<div class="bg-dna-wrap {side_class}"><div class="bg-dna-helix">{rungs}</div></div>'

DNA_3D_RUNGS = "".join(
    f'<div class="dna-rung" style="animation-delay:{round(-i * 0.28, 2)}s;"><span class="nuc-l"></span><span class="bond"></span><span class="nuc-r"></span></div>'
    for i in range(16)
)
DNA_3D_WIDGET = f'<div class="dna-stage"><div class="dna-helix">{DNA_3D_RUNGS}</div></div>'

BG_NEURON_SVG = """
<svg class="bg-neuron-canvas" viewBox="0 0 1440 900" preserveAspectRatio="xMidYMid slice" fill="none" xmlns="http://www.w3.org/2000/svg">
  <g stroke="rgba(56,189,248,0.18)" stroke-width="1.4">
    <path d="M120 180 L340 280 L560 190 L820 310 L1080 170 L1320 260"/>
    <path d="M80 540 L290 460 L520 620 L760 490 L1030 640 L1340 520"/>
    <path d="M340 280 L290 460 M560 190 L520 620 M820 310 L760 490 M1080 170 L1030 640"/>
    <path d="M220 760 L520 620 L880 780 L1220 710 L1340 520"/>
  </g>
  <g class="bg-axon-pulse" stroke="#00f5d4" stroke-width="2.2" stroke-linecap="round">
    <path d="M120 180 L340 280 L560 190 L820 310 L1080 170 L1320 260"/>
    <path d="M80 540 L290 460 L520 620 L760 490 L1030 640 L1340 520"/>
    <path d="M340 280 L290 460 M820 310 L760 490 M220 760 L520 620 L880 780"/>
  </g>
  <g class="bg-synapses">
    <circle cx="120" cy="180" r="5" fill="#00f5d4"/>
    <circle cx="340" cy="280" r="8" fill="#071426" stroke="#00f5d4" stroke-width="2.5"/>
    <circle cx="560" cy="190" r="6" fill="#38bdf8"/>
    <circle cx="820" cy="310" r="9" fill="#071426" stroke="#a78bfa" stroke-width="2.5"/>
    <circle cx="1080" cy="170" r="6" fill="#00f5d4"/>
    <circle cx="1320" cy="260" r="5" fill="#38bdf8"/>
    <circle cx="290" cy="460" r="7" fill="#071426" stroke="#38bdf8" stroke-width="2"/>
    <circle cx="520" cy="620" r="9" fill="#071426" stroke="#00f5d4" stroke-width="2.5"/>
    <circle cx="760" cy="490" r="6" fill="#a78bfa"/>
    <circle cx="1030" cy="640" r="8" fill="#071426" stroke="#38bdf8" stroke-width="2.5"/>
    <circle cx="1340" cy="520" r="6" fill="#00f5d4"/>
    <circle cx="880" cy="780" r="5" fill="#38bdf8"/>
  </g>
</svg>
"""

BG_3D_HTML = (
    '<div class="bio-bg-stage">'
    + BG_NEURON_SVG
    + make_bg_helix("bg-left", 24)
    + make_bg_helix("bg-right", 24)
    + '</div>'
)

NEURON_HEADER_SVG = """
<svg width="110" height="72" viewBox="0 0 120 78" fill="none" xmlns="http://www.w3.org/2000/svg">
  <path class="axon-pulse" d="M42 39 L12 16 M42 39 L8 39 M42 39 L14 62 M42 39 L44 10 M42 39 L78 39 L106 18 M78 39 L108 39 M78 39 L104 60" stroke="#00f5d4" stroke-width="2" stroke-linecap="round"/>
  <circle class="soma-glow" cx="42" cy="39" r="10" fill="#071426" stroke="#00f5d4" stroke-width="2.5"/>
  <circle cx="42" cy="39" r="4" fill="#38bdf8"/>
  <circle cx="78" cy="39" r="4.5" fill="#a78bfa"/>
  <circle cx="106" cy="18" r="3" fill="#00f5d4"/>
  <circle cx="108" cy="39" r="3" fill="#38bdf8"/>
  <circle cx="104" cy="60" r="3" fill="#a78bfa"/>
</svg>
"""

BIO_3D_CSS = """
<style>
@keyframes dnaSpin3D {
  0%   { transform: rotateY(0deg) scale(1); opacity: 0.92; }
  25%  { transform: rotateY(90deg) scale(0.84); opacity: 0.45; }
  50%  { transform: rotateY(180deg) scale(1); opacity: 0.92; }
  75%  { transform: rotateY(270deg) scale(0.84); opacity: 0.45; }
  100% { transform: rotateY(360deg) scale(1); opacity: 0.92; }
}
@keyframes axonFlow {
  0%   { stroke-dashoffset: 80; opacity: 0.25; }
  50%  { stroke-dashoffset: 0; opacity: 0.95; }
  100% { stroke-dashoffset: -80; opacity: 0.25; }
}
@keyframes somaPulse {
  0%, 100% { filter: drop-shadow(0 0 4px #00f5d4); }
  50%      { filter: drop-shadow(0 0 14px #38bdf8); }
}
@keyframes bgFloat {
  0%, 100% { transform: translateY(0px) rotateZ(-14deg); }
  50%      { transform: translateY(-18px) rotateZ(-10deg); }
}
/* Full-screen fixed 3D DNA + Neuron background behind the entire Gradio app */
.bio-bg-stage { position: fixed; inset: 0; z-index: 0; pointer-events: none; overflow: hidden; background: radial-gradient(circle at 20% 20%, #0d2240 0%, #050c1a 55%, #02060e 100%); }
.bg-neuron-canvas { position: absolute; inset: 0; width: 100%; height: 100%; opacity: 0.82; }
.bg-axon-pulse { stroke-dasharray: 18 36; animation: axonFlow 3.4s linear infinite; }
.bg-synapses circle { animation: somaPulse 3s ease-in-out infinite; }
.bg-dna-wrap { position: absolute; top: 6%; width: 150px; height: 88vh; perspective: 950px; opacity: 0.42; animation: bgFloat 8s ease-in-out infinite; }
.bg-dna-wrap.bg-left { left: 2%; }
.bg-dna-wrap.bg-right { right: 2%; animation-delay: -4s; }
.bg-dna-helix { width: 100%; height: 100%; display: flex; flex-direction: column; justify-content: space-between; transform-style: preserve-3d; }
.bg-dna-rung { display: flex; align-items: center; justify-content: space-between; width: 100%; height: 4px; transform-style: preserve-3d; animation: dnaSpin3D 6.2s linear infinite; }
.bg-nuc-l { width: 11px; height: 11px; border-radius: 50%; background: #00f5d4; box-shadow: 0 0 14px #00f5d4; }
.bg-nuc-r { width: 11px; height: 11px; border-radius: 50%; background: #38bdf8; box-shadow: 0 0 14px #38bdf8; }
.bg-bond { flex: 1; height: 2px; margin: 0 4px; background: linear-gradient(90deg, rgba(0,245,212,0.85), rgba(167,139,250,0.65), rgba(56,189,248,0.85)); }
/* Header mini 3D helix */
.dna-stage { width: 64px; height: 82px; perspective: 650px; display: flex; align-items: center; justify-content: center; }
.dna-helix { width: 52px; height: 78px; display: flex; flex-direction: column; justify-content: space-between; transform-style: preserve-3d; transform: rotateZ(-10deg); }
.dna-rung { display: flex; align-items: center; justify-content: space-between; width: 100%; height: 3px; transform-style: preserve-3d; animation: dnaSpin3D 4.2s linear infinite; }
.nuc-l { width: 7px; height: 7px; border-radius: 50%; background: #00f5d4; box-shadow: 0 0 9px #00f5d4; }
.nuc-r { width: 7px; height: 7px; border-radius: 50%; background: #38bdf8; box-shadow: 0 0 9px #38bdf8; }
.bond { flex: 1; height: 1.5px; margin: 0 2px; background: linear-gradient(90deg, rgba(0,245,212,0.8), rgba(167,139,250,0.6), rgba(56,189,248,0.8)); }
.axon-pulse { stroke-dasharray: 12 18; animation: axonFlow 2.2s linear infinite; }
.soma-glow { animation: somaPulse 2.4s ease-in-out infinite; }
/* Frosted glass foreground so the 3D DNA/Neuron background shows through */
.gradio-container { background: transparent !important; color: #e2e8f0 !important; position: relative; }
.gradio-container > .main, .gradio-container .wrap, .gradio-container .contain { position: relative; z-index: 2; }
.gradio-container .block, .gradio-container .panel, .gradio-container .form { background: rgba(7, 17, 34, 0.72) !important; backdrop-filter: blur(10px) !important; -webkit-backdrop-filter: blur(10px) !important; border: 1px solid rgba(56, 189, 248, 0.25) !important; border-radius: 14px !important; }
.gradio-container label span { color: #93c5fd !important; font-weight: 600 !important; }
.gradio-container button.primary { background: linear-gradient(135deg, #00b4d8, #00f5d4) !important; color: #04101d !important; font-weight: 800 !important; border: none !important; box-shadow: 0 0 18px rgba(0, 245, 212, 0.35) !important; }
.bio-card-3d { transition: transform 0.25s ease, box-shadow 0.25s ease; }
.bio-card-3d:hover { transform: translateY(-2px); box-shadow: 0 10px 24px rgba(0,245,212,0.22); }
</style>
"""

def compute_live_test_metrics(model_choice: str, thresh: float) -> dict:
    """Exact Accuracy, Precision, Recall, F1 (Parkinson class + Macro avg) & CM counts on test set."""
    if model_choice == "Ensemble (All 3 Models)":
        probs = np.mean([RESULTS[m]["y_prob"] for m in MODELS], axis=0)
    else:
        probs = RESULTS[model_choice]["y_prob"]
    preds = (probs >= thresh).astype(int)
    cm = confusion_matrix(y_true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, preds, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, preds, pos_label=1, zero_division=0),
        "macro_prec": precision_score(y_true, preds, average="macro", zero_division=0),
        "macro_rec": recall_score(y_true, preds, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }

def build_metrics_cards_html(model_choice: str, thresh: float) -> str:
    m = compute_live_test_metrics(model_choice, thresh)
    cards = [
        ("Accuracy", m["accuracy"], "#38bdf8", f"TP={m['tp']} TN={m['tn']} / N={len(y_true)}"),
        ("Precision (PD)", m["precision"], "#a78bfa", f"Macro Avg: {m['macro_prec']*100:.1f}% (FP={m['fp']})"),
        ("Recall (PD)", m["recall"], "#fb7185", f"Macro Avg: {m['macro_rec']*100:.1f}% (FN={m['fn']})"),
        ("F1 Score (PD)", m["f1"], "#00f5d4", f"Macro Avg: {m['macro_f1']*100:.1f}%"),
    ]
    items_html = ""
    for label, val, col, sub in cards:
        items_html += f"""
        <div class="bio-card-3d" style="background:rgba(7,17,34,0.78); backdrop-filter:blur(10px); border:1px solid {col}66; border-radius:14px; padding:12px 14px;">
          <div style="font-size:11px; text-transform:uppercase; letter-spacing:1px; color:#94a3b8; font-weight:700;">{label}</div>
          <div style="font-size:24px; font-weight:800; color:{col}; margin:3px 0;">{val * 100:.1f}%</div>
          <div style="height:5px; background:rgba(148,163,184,0.16); border-radius:999px; overflow:hidden; margin-bottom:5px;">
            <div style="width:{val * 100:.1f}%; height:100%; background:{col}; box-shadow:0 0 8px {col};"></div>
          </div>
          <div style="font-size:10px; color:#94a3b8;">{sub}</div>
        </div>
        """
    return f"""
    <div style="font-family:system-ui,sans-serif; margin-bottom:8px;">
      <div style="font-size:11px; color:#7dd3fc; text-transform:uppercase; letter-spacing:1px; margin-bottom:6px; font-weight:700;">
        Exact Held-Out Test Metrics &bull; {model_choice} @ Threshold {thresh:.2f}
      </div>
      <div style="display:grid; grid-template-columns:repeat(4, minmax(0, 1fr)); gap:10px;">{items_html}</div>
    </div>
    """

def build_banner_html(prob_p: float, thresh: float, mode_label: str) -> str:
    is_pos = prob_p >= thresh
    accent = "#fb7185" if is_pos else "#00f5d4"
    glow = "rgba(251, 113, 133, 0.25)" if is_pos else "rgba(0, 245, 212, 0.22)"
    title = "MOTOR ANOMALY — Parkinson's Pattern" if is_pos else "HEALTHY NEURAL MOTOR SIGNAL"
    pct_p, pct_h = prob_p * 100.0, (1.0 - prob_p) * 100.0
    lit_bars = max(1, min(12, round(prob_p * 12)))
    rungs = ""
    for i in range(12):
        active = i < lit_bars
        bar_col = ("#fb7185" if is_pos else "#00f5d4") if active else "rgba(148,163,184,0.18)"
        bar_glow = f"0 0 8px {accent}" if active else "none"
        rungs += f'<div style="flex:1; height:{12 + (i % 3) * 6}px; border-radius:4px; background:{bar_col}; box-shadow:{bar_glow};"></div>'

    return f"""
    <div style="border:1.5px solid {accent}; background:rgba(8,19,36,0.82); backdrop-filter:blur(10px); box-shadow:0 0 24px {glow}; border-radius:16px; padding:16px 18px; color:#e2e8f0; font-family:system-ui,sans-serif;">
      <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">
        <span style="font-size:17px; font-weight:800; color:{accent};">{title}</span>
        <span style="font-size:12px; padding:4px 11px; border-radius:999px; background:{glow}; border:1px solid {accent}; color:{accent}; font-weight:700;">
          {mode_label} &bull; Thresh {thresh:.2f}
        </span>
      </div>
      <div style="margin-top:12px; display:grid; grid-template-columns:1fr 1fr; gap:12px;">
        <div style="background:rgba(255,255,255,0.03); padding:10px 12px; border-radius:10px; border:1px solid rgba(251,113,133,0.22);">
          <div style="font-size:11px; text-transform:uppercase; color:#94a3b8;">Parkinson's Signal</div>
          <div style="font-size:24px; font-weight:800; color:#fb7185;">{pct_p:.1f}%</div>
        </div>
        <div style="background:rgba(255,255,255,0.03); padding:10px 12px; border-radius:10px; border:1px solid rgba(0,245,212,0.22);">
          <div style="font-size:11px; text-transform:uppercase; color:#94a3b8;">Healthy Motor Signal</div>
          <div style="font-size:24px; font-weight:800; color:#00f5d4;">{pct_h:.1f}%</div>
        </div>
      </div>
      <div style="margin-top:10px;">
        <div style="font-size:11px; color:#94a3b8; margin-bottom:5px;">Synaptic Activation Spectrum</div>
        <div style="display:flex; align-items:center; gap:5px; padding:6px 8px; background:rgba(5,11,22,0.65); border-radius:10px;">{rungs}</div>
      </div>
    </div>
    """

def build_consensus_html(per_model_probs: dict, thresh: float) -> str:
    rows_html = ""
    for m_name, p in per_model_probs.items():
        verdict = "Parkinson" if p >= thresh else "Healthy"
        badge_col = "#fb7185" if p >= thresh else "#00f5d4"
        m_stats = compute_live_test_metrics(m_name, thresh)
        rows_html += f"""
        <tr style="border-bottom:1px solid rgba(56,189,248,0.12);">
          <td style="padding:8px 6px; font-weight:700; color:#f1f5f9;">&#9679; {m_name}</td>
          <td style="padding:8px 6px; color:#7dd3fc;">{m_stats['accuracy']:.3f}</td>
          <td style="padding:8px 6px; color:#c4b5fd;">{m_stats['precision']:.3f}</td>
          <td style="padding:8px 6px; color:#fda4af;">{m_stats['recall']:.3f}</td>
          <td style="padding:8px 6px; color:#5eead4; font-weight:700;">{m_stats['f1']:.3f}</td>
          <td style="padding:8px 6px; width:26%;">
            <div style="display:flex; align-items:center; gap:6px;">
              <div style="flex:1; height:7px; background:rgba(148,163,184,0.16); border-radius:999px; overflow:hidden;">
                <div style="width:{p*100:.1f}%; height:100%; background:{badge_col}; box-shadow:0 0 8px {badge_col};"></div>
              </div>
              <span style="font-variant-numeric:tabular-nums; font-weight:700; color:#e2e8f0;">{p*100:.1f}%</span>
            </div>
          </td>
          <td style="padding:8px 6px; text-align:right;">
            <span style="color:#050b16; background:{badge_col}; padding:2px 9px; border-radius:999px; font-size:11px; font-weight:800;">{verdict}</span>
          </td>
        </tr>
        """
    return f"""
    <div style="background:rgba(7,17,34,0.78); backdrop-filter:blur(10px); border:1px solid rgba(56,189,248,0.25); border-radius:14px; padding:14px 16px; color:#e2e8f0; font-family:system-ui,sans-serif;">
      <div style="font-weight:700; margin-bottom:8px; font-size:13px; color:#38bdf8; text-transform:uppercase; letter-spacing:0.8px;">Neural Pathway Consensus &amp; Exact Test Metrics (@ Thresh {thresh:.2f})</div>
      <table style="width:100%; border-collapse:collapse; font-size:12px;">
        <thead>
          <tr style="text-align:left; color:#94a3b8; border-bottom:1px solid rgba(56,189,248,0.2);">
            <th style="padding:6px;">Pathway</th>
            <th style="padding:6px;">Acc</th>
            <th style="padding:6px;">Prec</th>
            <th style="padding:6px;">Rec</th>
            <th style="padding:6px;">F1</th>
            <th style="padding:6px;">P(PD)</th>
            <th style="padding:6px; text-align:right;">Call</th>
          </tr>
        </thead>
        <tbody>{rows_html}</tbody>
      </table>
    </div>
    """

def screen_image_pro(pil_img: Image.Image, model_choice: str, thresh: float, cam_alpha: float):
    metrics_html = build_metrics_cards_html(model_choice, thresh)
    if pil_img is None:
        empty_msg = "<div style='padding:20px; border-radius:14px; background:rgba(7,17,34,0.72); border:1px dashed rgba(0,245,212,0.35); text-align:center; color:#94a3b8;'>Upload a spiral/wave drawing or select a test sample below to activate neural diagnostics.</div>"
        return None, None, None, metrics_html, empty_msg, ""

    resized = ImageOps.exif_transpose(pil_img).convert("RGB").resize(TARGET_SIZE)
    arr_255 = np.asarray(resized, dtype=np.float32)
    batch = np.expand_dims(arr_255, 0)

    per_model_probs = {name: float(m.predict(batch, verbose=0)[0, 0]) for name, m in MODELS.items()}

    if model_choice == "Ensemble (All 3 Models)":
        prob_p = float(np.mean(list(per_model_probs.values())))
        cam_model = best_model_name
        mode_label = f"3-Pathway Ensemble ({cam_model} CAM)"
    else:
        prob_p = per_model_probs[model_choice]
        cam_model = model_choice
        mode_label = model_choice

    heatmap = compute_gradcam(arr_255, cam_model)
    overlay = overlay_gradcam(arr_255, heatmap, alpha=cam_alpha)
    pure_cam = (plt.get_cmap("jet")(heatmap)[..., :3] * 255.0).astype("uint8")

    return Image.fromarray(overlay), Image.fromarray(pure_cam), resized, metrics_html, build_banner_html(prob_p, thresh, mode_label), build_consensus_html(per_model_probs, thresh)

with gr.Blocks(title="NeuroHelix 3D AI — Parkinson's Neuron & DNA Screening") as demo:
    gr.HTML(BIO_3D_CSS + BG_3D_HTML + f"""
    <div style="background:rgba(7,17,34,0.78); backdrop-filter:blur(12px); border:1px solid rgba(0,245,212,0.32); box-shadow:0 8px 32px rgba(0,245,212,0.16); color:#f8fafc; padding:18px 24px; border-radius:18px; margin-bottom:14px; font-family:system-ui,sans-serif;">
      <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:14px;">
        <div style="display:flex; align-items:center; gap:16px;">
          {DNA_3D_WIDGET}
          <div>
            <div style="font-size:11px; text-transform:uppercase; letter-spacing:1.6px; color:#00f5d4; font-weight:700;">3D Bio-Neural Diagnostics Engine</div>
            <h1 style="margin:2px 0 0; font-size:23px; font-weight:800; color:#ffffff;">NeuroHelix 3D &mdash; Parkinson's Motor Screening</h1>
            <p style="margin:4px 0 0; color:#94a3b8; font-size:13px;">Live Accuracy, Precision, Recall &amp; F1 Telemetry + Synaptic Grad-CAM</p>
          </div>
        </div>
        <div style="display:flex; align-items:center; gap:14px; flex-wrap:wrap;">
          {NEURON_HEADER_SVG}
          <div style="display:flex; flex-direction:column; gap:6px;">
            <span style="background:rgba(0,245,212,0.12); border:1px solid rgba(0,245,212,0.35); color:#00f5d4; padding:4px 11px; border-radius:999px; font-size:12px;">Lead Network: <b>{best_model_name}</b></span>
            <span style="background:rgba(56,189,248,0.12); border:1px solid rgba(56,189,248,0.35); color:#7dd3fc; padding:4px 11px; border-radius:999px; font-size:12px;">F1: <b>{RESULTS[best_model_name]['f1']:.3f}</b> &bull; Recall: <b>{RESULTS[best_model_name]['recall']:.3f}</b></span>
          </div>
        </div>
      </div>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("3D Synaptic Screening & Live Metrics"):
            metrics_strip_out = gr.HTML(build_metrics_cards_html(best_model_name, DEFAULT_THRESH))
            with gr.Row():
                with gr.Column(scale=5):
                    in_img = gr.Image(type="pil", label="Motor Drawing Input (Spiral / Wave)", height=280)
                    model_dd = gr.Dropdown(choices=MODEL_CHOICES, value=best_model_name, label="Neural Pathway Mode")
                    with gr.Row():
                        thresh_sl = gr.Slider(0.10, 0.90, value=DEFAULT_THRESH, step=0.01, label="Decision Threshold (0.50 = Standard)")
                        alpha_sl = gr.Slider(0.15, 0.80, value=0.45, step=0.05, label="Heatmap Glow")
                    with gr.Row():
                        run_btn = gr.Button("Run 3D Neural Scan", variant="primary")
                        opt_btn = gr.Button(f"Use Val-Tuned Thresh ({OPT_THRESH:.2f})")
                    if example_paths:
                        gr.Examples(examples=example_paths, inputs=[in_img], label="Test Cohort Samples (Click to Scan)")

                with gr.Column(scale=7):
                    banner_out = gr.HTML("<div style='padding:20px; border-radius:14px; background:rgba(7,17,34,0.72); border:1px dashed rgba(0,245,212,0.35); text-align:center; color:#94a3b8;'>Upload a drawing or select a test sample to activate neural diagnostics.</div>")
                    with gr.Tabs():
                        with gr.Tab("Grad-CAM Synapse Overlay"):
                            cam_out = gr.Image(type="pil", label="Spatial Activation Overlay", height=280)
                        with gr.Tab("Pure Attention Field"):
                            heat_out = gr.Image(type="pil", label="Saliency Heatmap", height=280)
                        with gr.Tab("224x224 Input"):
                            pre_out = gr.Image(type="pil", label="Standardized Input", height=280)
                    consensus_out = gr.HTML()

            inputs_all = [in_img, model_dd, thresh_sl, alpha_sl]
            outputs_all = [cam_out, heat_out, pre_out, metrics_strip_out, banner_out, consensus_out]
            opt_btn.click(lambda: OPT_THRESH, outputs=[thresh_sl])
            run_btn.click(screen_image_pro, inputs=inputs_all, outputs=outputs_all)
            in_img.change(screen_image_pro, inputs=inputs_all, outputs=outputs_all)
            model_dd.change(screen_image_pro, inputs=inputs_all, outputs=outputs_all)
            thresh_sl.change(screen_image_pro, inputs=inputs_all, outputs=outputs_all)
            alpha_sl.change(screen_image_pro, inputs=inputs_all, outputs=outputs_all)

        with gr.Tab("Cohort Benchmark & Diagnostics"):
            gr.Dataframe(value=table.reset_index().rename(columns={"index": "Model"}).round(3), label="Held-Out Test Metrics (@ Threshold 0.50)", interactive=False)
            with gr.Row():
                gr.Image("metrics_comparison.png", label="Metrics Comparison")
                gr.Image("roc_curves.png", label="ROC Curves")
            with gr.Row():
                gr.Image("confusion_matrices.png", label="Confusion Matrices")
                gr.Image("gradcam_analysis.png", label="Test Set Grad-CAM Grid")

    gr.HTML("<p style='text-align:center; font-size:12px; color:#94a3b8; margin-top:10px;'>NeuroHelix 3D AI &bull; Research &amp; educational screening prototype only — not a medical device. Clinical diagnosis requires a qualified neurologist.</p>")

demo.launch(share=True, debug=False)